In [ ]:
from mnist import Net, test
from hyptorch.pmath import dist0

%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib import cm

import torch
from torchvision import datasets, transforms
import numpy as np
import os
import json
import dataclasses as dc

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

np.random.seed(0)
torch.manual_seed(0)

state_dict_path = "checkpoints/mnist_cnn_2.pt"
args_path = "args.json"
assert os.path.isfile(state_dict_path)
assert os.path.isfile(args_path)

In [ ]:
save_imgs = True
if save_imgs:
    os.makedirs("exp_images", exist_ok=True)

In [ ]:
with open(args_path, "r") as f:
    args = json.load(f)
args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)

In [ ]:
model = Net(args).to(device)

state_dict = torch.load(state_dict_path, map_location=device)
model.load_state_dict(state_dict)

In [ ]:
kwargs = {"num_workers": 1, "pin_memory": True} if device.type == "cuda" else {}
train_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        "../data",
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.batch_size,
    shuffle=True,
    **kwargs
)

test_MNIST_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        "../data",
        train=False,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
test_Omniglot_loader = torch.utils.data.DataLoader(
    datasets.Omniglot(
        "../data",
        background=False,
        download=True,
        transform=transforms.Compose(
            [
                transforms.Resize((28, 28)), 
                transforms.ToTensor(),
                transforms.Lambda(lambda image: 1.0 - image),
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
omni_data, omni_target = next(iter(test_Omniglot_loader))
omni_data, omni_target = omni_data.to(device), omni_target.to(device)

num_samples = 20
omni_sample_data = omni_data.detach().cpu().numpy()[:num_samples]
omni_sample_target = omni_target.detach().cpu().numpy()[:num_samples]
omni_sample_embeddings = model.embed(omni_data)[:num_samples]

In [ ]:
data, target = next(iter(test_MNIST_loader))
data, target = data.to(device), target.to(device)
output = model(data)

num_samples = 40
sample_data = data.detach().cpu().numpy()[:num_samples]
sample_target = target.detach().cpu().numpy()[:num_samples]
sample_embeddings = model.embed(data)[:num_samples]

In [ ]:
fig, ax = plt.subplots(10//2, 2, figsize=(6, 8))

for i in range(10):
    ax[i//2, i%2].imshow(sample_data[i, 0], aspect="equal", cmap="gray")
    ax[i//2, i%2].axis("off")

vals = np.exp(model(data).detach().cpu().numpy()[:10])
sample_pred = np.argmax(vals, axis=1)
sample_conf = np.max(vals, axis=1)
for i in range(10):
    ax[i//2, i%2].set_title(f"Pred: {sample_pred[i]}\nConf: {sample_conf[i]:.2f}")

plt.tight_layout()
plt.show()

### Figure 1 from the paper

In [ ]:
np_sample_embeddings = sample_embeddings.detach().cpu().numpy()
np_omni_sample_embeddings = omni_sample_embeddings.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 10))

circle = plt.Circle((0, 0), 1, color="black", fill=False)
ax.add_artist(circle)

colors = cm.rainbow(np.linspace(0, 1, 10))
for i, (x, y) in enumerate(np_sample_embeddings):
    img = sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color=colors[sample_target[i]], s=50)

for i, (x, y) in enumerate(np_omni_sample_embeddings):
    img = omni_sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color="black", s=50, marker="x")

ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")

plt.axis("off")
if save_imgs:
    plt.savefig("exp_images/embedding_plot.png")
else:
    plt.show()

### Compare distributions

In [ ]:
MNIST_dist = np.array([])

for x in test_MNIST_loader:
    data, target = x
    data, target = data.to(device), target.to(device)

    embeddings = model.embed(data)
    dist_to_0 = dist0(embeddings).detach().cpu().numpy()
    MNIST_dist = np.concatenate((MNIST_dist, dist_to_0))

In [ ]:
omni_dist = np.array([])

for x in test_Omniglot_loader:
    data, target = x
    data, target = data.to(device), target.to(device)

    embeddings = model.embed(data)
    dist_to_0 = dist0(embeddings).detach().cpu().numpy()
    omni_dist = np.concatenate((omni_dist, dist_to_0))

In [ ]:
fig, ax = plt.subplots() 

ax.hist(MNIST_dist, bins=50, alpha=0.4, color='red', density=True, label='MNIST')
ax.hist(omni_dist, bins=50, alpha=0.4, color='blue', density=True, label='Omniglot')
ax.set_xlabel('Distance to 0')
ax.set_ylabel('Density')
ax.set_title('Histogram of Distances to 0')
ax.legend()

plt.show()